# Orthogroup-level ALP differential-expression / AEP cell-type enrichment with broad-expression filtering and gene–cell-type export

This self-contained analysis maps *Hydra oligactis* ALP DESeq2 genes to OrthoFinder orthogroups, associates those orthogroups with *Hydra vulgaris* AEP cell-type expression, and tests whether up- or downregulated orthogroups are overrepresented among highly expressed AEP orthogroups.

**Statistical unit:** one orthogroup. Before aggregation, mapped ALP DESeq2 genes assigned to more than the configured maximum number of high-expression AEP cell types are excluded. **Threshold universe:** all orthogroups with mapped AEP expression, independent of DESeq2 mapping. **Fisher-test universe:** orthogroups with mapped AEP expression and at least one unambiguous mapped ALP DESeq2 gene. One-sided Fisher tests assess overrepresentation; Benjamini–Hochberg correction is applied jointly across every direction-by-cell-type test.

The notebook also exports one row per assigned gene–cell-type pair; unassigned pairs are intentionally omitted. The final interpretation note explains the analysis units and background.

## Configuration

`HIGH_EXPRESSION_QUANTILE = 0.90` implements the top 10% definition. Ambiguous ALP genes are excluded by default (or can trigger an error). The broad-expression filter excludes genes assigned to more than the configured `MAX_ASSIGNED_CELL_TYPES` value (currently 1). Orthogroups significant in both directions are excluded from both sets by default (or can be included in both). Genes missing either `log2FoldChange` or `padj` are excluded from the enrichment background by default. The gene–cell-type export contains assigned pairs only; it does not expand unassigned pairs.

In [ ]:
from pathlib import Path
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
CELLTYPE_EXPRESSION_FILE = PROJECT_ROOT / "data" / "genes_to_cell_type_table.tsf"
ORTHOGROUP_FILE = PROJECT_ROOT / "data" / "hydra_aep_oligactis" / "Results_Jun11" / "Orthogroups" / "Orthogroups.tsv"
DESEQ_FILE = PROJECT_ROOT / "data" / "alp_deseq2_output.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "results" / "orthogroup_expression_celltype_enrichment_gene_celltype"
PADJ_CUTOFF = 0.05
MIN_ABS_LOG2_FOLD_CHANGE = 0.0
HIGH_EXPRESSION_QUANTILE = 0.90
MAX_ASSIGNED_CELL_TYPES = 1
FDR_CUTOFF = 0.05
AMBIGUOUS_GENE_POLICY = "exclude"  # "exclude" or "error"
BOTH_DIRECTIONS_POLICY = "exclude" # "exclude" or "include_both"
MISSING_DE_STATS_BACKGROUND_POLICY = "exclude" # "exclude" or "include_as_not_significant"
GENE_CELLTYPE_INCLUDE_UNASSIGNED = False # fixed false: assigned pairs only
CORRECTION_FAMILY = "Benjamini-Hochberg across all direction-by-cell-type tests"

if not 0 < HIGH_EXPRESSION_QUANTILE < 1: raise ValueError("HIGH_EXPRESSION_QUANTILE must be in (0,1)")
if not isinstance(MAX_ASSIGNED_CELL_TYPES, int) or MAX_ASSIGNED_CELL_TYPES < 0: raise ValueError("MAX_ASSIGNED_CELL_TYPES must be a non-negative integer")
if AMBIGUOUS_GENE_POLICY not in {"exclude","error"}: raise ValueError("Invalid AMBIGUOUS_GENE_POLICY")
if BOTH_DIRECTIONS_POLICY not in {"exclude","include_both"}: raise ValueError("Invalid BOTH_DIRECTIONS_POLICY")
if MISSING_DE_STATS_BACKGROUND_POLICY not in {"exclude","include_as_not_significant"}: raise ValueError("Invalid MISSING_DE_STATS_BACKGROUND_POLICY")
if GENE_CELLTYPE_INCLUDE_UNASSIGNED: raise ValueError("This notebook exports assigned gene-cell-type pairs only")
for p in [CELLTYPE_EXPRESSION_FILE, ORTHOGROUP_FILE, DESEQ_FILE]:
    if not p.is_file(): raise FileNotFoundError(p)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Root: {PROJECT_ROOT}")
print(f"High-expression quantile={HIGH_EXPRESSION_QUANTILE:.2f} (top {(1-HIGH_EXPRESSION_QUANTILE)*100:.0f}%)")
print(f"Broad-expression filter: retain genes assigned to <= {MAX_ASSIGNED_CELL_TYPES} cell types")
print(f"DE: padj < {PADJ_CUTOFF}, |log2FC| >= {MIN_ABS_LOG2_FOLD_CHANGE}; FDR < {FDR_CUTOFF}")
print(f"Policies: ambiguous genes={AMBIGUOUS_GENE_POLICY}; both directions={BOTH_DIRECTIONS_POLICY}")
print(f"Missing DE statistics in background: {MISSING_DE_STATS_BACKGROUND_POLICY}")
print("Gene-cell-type export: assigned pairs only; unassigned pairs omitted")

## Input loading and validation

The AEP file has an unnamed ID field and is read with that field as its index. The workbook's single-column embedded TSV is detected explicitly. Invalid numeric values are reported and never replaced with zero.

In [ ]:
def require_columns(df, cols, label):
    missing = sorted(set(cols)-set(df.columns))
    if missing: raise ValueError(f"{label} missing columns: {missing}")

orthogroups = pd.read_csv(ORTHOGROUP_FILE, sep="\t", dtype=str)
require_columns(orthogroups, ["Orthogroup","aep","oligactis"], "Orthogroups")
if orthogroups["Orthogroup"].isna().any() or orthogroups["Orthogroup"].duplicated().any():
    raise ValueError("Orthogroup IDs must be present and unique")

expression_raw = pd.read_csv(CELLTYPE_EXPRESSION_FILE, sep="\t", index_col=0, dtype=str)
expression_raw.index = expression_raw.index.astype(str).str.strip()
expression_raw.index.name = "expression_aep_id"
if expression_raw.index.duplicated().any(): raise ValueError("Duplicate AEP expression IDs")
celltypes = expression_raw.columns.tolist()
expression = expression_raw.apply(pd.to_numeric, errors="coerce")
bad_expr = expression.isna() & expression_raw.notna()
nonfinite_expr = pd.DataFrame(~np.isfinite(expression.to_numpy(float)), index=expression.index, columns=celltypes) & expression.notna()
expr_issue_mask = expression_raw.isna() | bad_expr | nonfinite_expr
expr_issue_index = expr_issue_mask.stack()[lambda x: x].index
invalid_expression_values = pd.DataFrame({
    "expression_aep_id": expr_issue_index.get_level_values(0),
    "celltype": expr_issue_index.get_level_values(1),
    "raw_value": [expression_raw.at[i, c] for i, c in expr_issue_index],
    "issue": ["missing" if pd.isna(expression_raw.at[i,c]) else "invalid_or_nonfinite" for i,c in expr_issue_index],
})
expression = expression.mask(nonfinite_expr)

deseq_raw = pd.read_excel(DESEQ_FILE, dtype=str)
if deseq_raw.shape[1] == 1 and "\t" in str(deseq_raw.columns[0]):
    names = str(deseq_raw.columns[0]).split("\t")
    deseq_raw = deseq_raw.iloc[:,0].fillna("").str.split("\t", expand=True)
    if deseq_raw.shape[1] != len(names): raise ValueError("Malformed embedded DESeq2 TSV")
    deseq_raw.columns = names
required_de = ["transcriptID","baseMean","log2FoldChange","lfcSE","stat","pvalue","padj"]
require_columns(deseq_raw, required_de, "DESeq2")
deseq_raw["transcriptID"] = deseq_raw["transcriptID"].astype("string").str.strip()
if deseq_raw["transcriptID"].isna().any() or deseq_raw["transcriptID"].eq("").any() or deseq_raw["transcriptID"].duplicated().any():
    raise ValueError("DESeq2 transcriptID must be nonblank and unique")
deseq = deseq_raw.copy()
bad_de = []
for col in required_de[1:]:
    num = pd.to_numeric(deseq_raw[col], errors="coerce")
    missing = deseq_raw[col].isna() | deseq_raw[col].eq("")
    invalid = num.isna() & ~missing
    nonfinite = num.notna() & ~np.isfinite(num)
    issue = missing | invalid | nonfinite
    if issue.any():
        x=deseq_raw.loc[issue,["transcriptID",col]].rename(columns={col:"raw_value"})
        x["field"]=col
        x["issue"]=np.select([missing[issue],invalid[issue],nonfinite[issue]],["missing","invalid","nonfinite"], default="unknown")
        bad_de.append(x)
    deseq[col] = num.mask(nonfinite)
invalid_deseq_values = pd.concat(bad_de, ignore_index=True) if bad_de else pd.DataFrame(columns=["transcriptID","raw_value","field","issue"])
print(f"Inputs: {len(orthogroups):,} orthogroups; {len(expression):,} AEP IDs x {len(celltypes)} cell types; {len(deseq):,} DESeq2 genes")
print(f"Missing/invalid expression cells={len(invalid_expression_values):,}; missing/invalid DESeq2 numeric cells={len(invalid_deseq_values["transcriptID"].unique()):,}")

In [ ]:
deseq_raw[deseq_raw["transcriptID"].isin(invalid_deseq_values["transcriptID"])]

## ID mapping

AEP `HVAEP1-G######` IDs are mapped to OrthoFinder `HVAEP#.G######` IDs by a validated, globally unique terminal `G######` token. ALP protein suffixes matching `_i<number>.p<number>` are removed. Multi-orthogroup genes are reported and never counted in multiple groups.

In [ ]:
def explode_members(df, species_col, id_col):
    x=df[["Orthogroup",species_col]].dropna().copy()
    x[id_col]=x[species_col].str.split(","); x=x.explode(id_col).drop(columns=species_col)
    x[id_col]=x[id_col].astype(str).str.strip()
    return x[x[id_col].ne("")].drop_duplicates()

aep_members=explode_members(orthogroups,"aep","aep_protein_id")
alp_members=explode_members(orthogroups,"oligactis","alp_protein_id")

# Validate complete IDs before extracting the unique terminal mapping token.
AEP_EXPRESSION_PATTERN=r"^HVAEP\d+-(G\d{6})$"
AEP_ORTHOFINDER_PATTERN=r"^HVAEP\d+\.(G\d{6})$"
aep_members["token"]=aep_members["aep_protein_id"].str.extract(AEP_ORTHOFINDER_PATTERN,expand=False)
expression_map=pd.DataFrame({"expression_aep_id":expression.index})
expression_map["token"]=expression_map["expression_aep_id"].str.extract(AEP_EXPRESSION_PATTERN,expand=False)
token_counts=aep_members.dropna(subset=["token"]).groupby("token")["aep_protein_id"].nunique()
duplicate_tokens=set(token_counts[token_counts.gt(1)].index)
lookup=aep_members[aep_members["token"].notna() & ~aep_members["token"].isin(duplicate_tokens)][["token","aep_protein_id"]].drop_duplicates()
expression_map=expression_map.merge(lookup,on="token",how="left")
expression_map["status"]=np.select(
    [expression_map["token"].isna(),expression_map["token"].isin(duplicate_tokens),expression_map["aep_protein_id"].isna()],
    ["invalid_full_id_pattern","ambiguous_token","unmapped"],default="mapped")
mapped_expression_map=expression_map[expression_map["status"].eq("mapped")].copy()
invalid_aep_orthofinder_ids=aep_members[aep_members["token"].isna()][["Orthogroup","aep_protein_id"]].copy()
if mapped_expression_map["aep_protein_id"].duplicated().any(): raise ValueError("AEP mapping is not one-to-one")

ALP_ISOFORM_PATTERN=r"^(.+)_i\d+\.p\d+$"
alp_members["alp_gene_id"]=alp_members["alp_protein_id"].str.extract(ALP_ISOFORM_PATTERN,expand=False)
invalid_alp_protein_ids=alp_members[alp_members["alp_gene_id"].isna()][["Orthogroup","alp_protein_id"]].copy()
invalid_alp_protein_ids["expected_pattern"]=ALP_ISOFORM_PATTERN
invalid_alp_protein_ids["diagnostic"]="failed_isoform_normalization"
alp_all=alp_members.dropna(subset=["alp_gene_id"])[["Orthogroup","alp_protein_id","alp_gene_id"]].drop_duplicates()
gene_og_count=alp_all.groupby("alp_gene_id")["Orthogroup"].nunique()
ambiguous_ids=set(gene_og_count[gene_og_count.gt(1)].index)
ambiguous_gene_mappings=alp_all[alp_all["alp_gene_id"].isin(ambiguous_ids)].sort_values(["alp_gene_id","Orthogroup"])
if ambiguous_ids and AMBIGUOUS_GENE_POLICY=="error": raise ValueError(f"{len(ambiguous_ids)} genes map to multiple orthogroups")
alp_map=alp_all[~alp_all["alp_gene_id"].isin(ambiguous_ids)][["Orthogroup","alp_gene_id"]].drop_duplicates()
de_ids=set(deseq["transcriptID"]); mapped_de_ids=de_ids & set(alp_map["alp_gene_id"])
unmapped_de=sorted(de_ids-set(alp_all["alp_gene_id"])); ambiguous_de=sorted(de_ids&ambiguous_ids)
id_mapping_diagnostics=pd.concat([
 expression_map.assign(source="AEP_expression",identifier=expression_map["expression_aep_id"],detail=expression_map["token"])[["source","identifier","status","detail"]],
 pd.DataFrame({"source":"AEP_OrthoFinder","identifier":invalid_aep_orthofinder_ids["aep_protein_id"],"status":"invalid_full_id_pattern","detail":""}),
 pd.DataFrame({"source":"ALP_DESeq2","identifier":unmapped_de,"status":"unmapped","detail":""}),
 pd.DataFrame({"source":"ALP_DESeq2","identifier":ambiguous_de,"status":"excluded_ambiguous","detail":""})],ignore_index=True)
print(f"AEP mapped {len(mapped_expression_map):,}/{len(expression_map):,} ({len(mapped_expression_map)/len(expression_map):.1%}); duplicate tokens={len(duplicate_tokens)}")
print(f"Invalid full AEP IDs: expression={int(expression_map.token.isna().sum()):,}; OrthoFinder={len(invalid_aep_orthofinder_ids):,}")
print(f"ALP protein IDs failing isoform normalization: {len(invalid_alp_protein_ids):,}")
print(f"DESeq2 mapped {len(mapped_de_ids):,}/{len(de_ids):,} ({len(mapped_de_ids)/len(de_ids):.1%}); unmapped={len(unmapped_de):,}; ambiguous/excluded={len(ambiguous_de):,}")

In [ ]:
ambiguous_de

## Valid-expression handling, thresholding, gene–cell-type assignments, filtering, and aggregation

Orthogroups whose cell-type means are all missing/non-finite are explicitly excluded from thresholding and enrichment and exported as diagnostics. Thresholds use every remaining orthogroup with mapped AEP expression, independently of DE significance. The realized high-expression fraction is reported because ties at an inclusive (`>=`) quantile threshold can yield more than the nominal top fraction.

Each unambiguous mapped ALP DESeq2 gene inherits assignments from its orthogroup. The gene–cell-type table exports **assigned pairs only**; unassigned pairs are omitted. Broadly expressed genes and, by default, genes missing either `log2FoldChange` or `padj` are removed before constructing orthogroup DE sets and the eligible background.

In [ ]:
mapped_expression=mapped_expression_map[["expression_aep_id","aep_protein_id"]].merge(expression.reset_index(),on="expression_aep_id",how="left")
og_expression_all=aep_members[["Orthogroup","aep_protein_id"]].merge(mapped_expression,on="aep_protein_id",how="inner").groupby("Orthogroup",as_index=False)[celltypes].mean()

# Explicitly remove orthogroups with no valid measurement in any cell type.
valid_measurement_count=og_expression_all[celltypes].notna().sum(axis=1)
orthogroups_without_valid_expression=og_expression_all.loc[valid_measurement_count.eq(0),["Orthogroup"]].copy()
orthogroups_without_valid_expression["reason"]="no_valid_expression_measurements"
og_expression=og_expression_all.loc[valid_measurement_count.gt(0)].copy().reset_index(drop=True)

threshold_rows=[]; flags=og_expression[["Orthogroup"]].copy()
for celltype in celltypes:
    values=og_expression[celltype].dropna()
    threshold=values.quantile(HIGH_EXPRESSION_QUANTILE) if len(values) else np.nan
    high=og_expression[celltype].ge(threshold) if np.isfinite(threshold) else pd.Series(False,index=og_expression.index)
    flags[celltype]=high
    high_count=int(high.sum()); denominator=len(values)
    threshold_rows.append({
        "celltype":celltype,
        "high_expression_quantile":HIGH_EXPRESSION_QUANTILE,
        "max_assigned_cell_types":MAX_ASSIGNED_CELL_TYPES,
        "nominal_high_expression_fraction":1-HIGH_EXPRESSION_QUANTILE,
        "threshold":threshold,
        "threshold_orthogroup_count":denominator,
        "highly_expressed_orthogroup_count_all_expression":high_count,
        "realized_high_expression_fraction":high_count/denominator if denominator else np.nan,
        "realized_minus_nominal_fraction":high_count/denominator-(1-HIGH_EXPRESSION_QUANTILE) if denominator else np.nan,
        "threshold_is_inclusive":"yes (>=); ties are included",
    })
threshold_table=pd.DataFrame(threshold_rows)

# Combine mapping with DE statistics and classify every mapped gene before filtering.
mapped_gene_de=(
    alp_map[alp_map["alp_gene_id"].isin(de_ids)].drop_duplicates(["alp_gene_id","Orthogroup"])
    .merge(deseq.rename(columns={"transcriptID":"alp_gene_id"})[["alp_gene_id","log2FoldChange","padj"]],on="alp_gene_id",how="left",validate="one_to_one")
)
mapped_gene_de["direction"]="not_significant"
valid_stats=mapped_gene_de["log2FoldChange"].notna()&mapped_gene_de["padj"].notna()
sig=valid_stats&mapped_gene_de["padj"].lt(PADJ_CUTOFF)&mapped_gene_de["log2FoldChange"].abs().ge(MIN_ABS_LOG2_FOLD_CHANGE)
mapped_gene_de.loc[sig&mapped_gene_de["log2FoldChange"].gt(0),"direction"]="upregulated"
mapped_gene_de.loc[sig&mapped_gene_de["log2FoldChange"].lt(0),"direction"]="downregulated"
mapped_gene_de.loc[~valid_stats,"direction"]="missing_statistics"

mapped_de_gene_map=mapped_gene_de.merge(flags,on="Orthogroup",how="left",validate="many_to_one")
for celltype in celltypes:
    mapped_de_gene_map[celltype]=mapped_de_gene_map[celltype].fillna(False).astype(bool)

def assigned_celltype_string(row):
    return ";".join(c for c in celltypes if bool(row[c]))

gene_filter_diagnostics=mapped_de_gene_map[["alp_gene_id","Orthogroup","log2FoldChange","padj","direction"]].copy()
gene_filter_diagnostics["assigned_cell_types"]=mapped_de_gene_map.apply(assigned_celltype_string,axis=1)
gene_filter_diagnostics["assigned_cell_type_count"]=mapped_de_gene_map[celltypes].sum(axis=1).astype(int)
gene_filter_diagnostics["high_expression_quantile"]=HIGH_EXPRESSION_QUANTILE
gene_filter_diagnostics["max_assigned_cell_types"]=MAX_ASSIGNED_CELL_TYPES
gene_filter_diagnostics["excluded"]=gene_filter_diagnostics["assigned_cell_type_count"].gt(MAX_ASSIGNED_CELL_TYPES)
gene_filter_diagnostics["exclusion_reason"]=np.where(gene_filter_diagnostics["excluded"],"assigned_cell_type_count_exceeds_maximum","retained_at_or_below_maximum")
gene_filter_diagnostics["has_required_de_statistics"]=valid_stats.to_numpy()
if MISSING_DE_STATS_BACKGROUND_POLICY=="exclude":
    stats_eligible=gene_filter_diagnostics["has_required_de_statistics"]
else:
    stats_eligible=pd.Series(True,index=gene_filter_diagnostics.index)
gene_filter_diagnostics["background_eligible_gene"]=(~gene_filter_diagnostics["excluded"])&stats_eligible
gene_filter_diagnostics["background_exclusion_reason"]=np.select(
    [gene_filter_diagnostics["excluded"],~stats_eligible],
    ["broad_expression_filter","missing_required_de_statistics"],default="eligible")
retained_gene_ids=set(gene_filter_diagnostics.loc[gene_filter_diagnostics["background_eligible_gene"],"alp_gene_id"])
retained_alp_map=alp_map[alp_map["alp_gene_id"].isin(retained_gene_ids)].copy()

# Assigned-pairs-only gene-cell-type table.
flag_long=mapped_de_gene_map.melt(
    id_vars=["alp_gene_id","Orthogroup","log2FoldChange","padj","direction"],
    value_vars=celltypes,var_name="celltype",value_name="high_expression_assigned")
flag_long=flag_long[flag_long["high_expression_assigned"]].copy()
expr_long=og_expression.melt(id_vars="Orthogroup",value_vars=celltypes,var_name="celltype",value_name="mean_aep_expression")
gene_celltype_table=(
    flag_long.merge(expr_long,on=["Orthogroup","celltype"],how="left",validate="many_to_one")
    .merge(threshold_table[["celltype","threshold"]],on="celltype",how="left",validate="many_to_one")
    .merge(gene_filter_diagnostics[["alp_gene_id","assigned_cell_type_count","excluded","exclusion_reason","background_eligible_gene","background_exclusion_reason"]],on="alp_gene_id",how="left",validate="many_to_one")
)
gene_celltype_table["unassigned_pairs_included"]=False
gene_celltype_table["high_expression_quantile"]=HIGH_EXPRESSION_QUANTILE
gene_celltype_table["max_assigned_cell_types"]=MAX_ASSIGNED_CELL_TYPES
gene_celltype_table=gene_celltype_table.sort_values(["alp_gene_id","celltype"]).reset_index(drop=True)

mapped_de=deseq.rename(columns={"transcriptID":"alp_gene_id"}).merge(retained_alp_map,on="alp_gene_id",how="inner")
mapped_de["direction"]="not_significant"
valid_for_classification=mapped_de["log2FoldChange"].notna()&mapped_de["padj"].notna()
sig=valid_for_classification&mapped_de["padj"].lt(PADJ_CUTOFF)&mapped_de["log2FoldChange"].abs().ge(MIN_ABS_LOG2_FOLD_CHANGE)
mapped_de.loc[sig&mapped_de["log2FoldChange"].gt(0),"direction"]="upregulated"
mapped_de.loc[sig&mapped_de["log2FoldChange"].lt(0),"direction"]="downregulated"

og_de=mapped_de.groupby("Orthogroup").agg(
    alp_deseq_gene_count=("alp_gene_id","nunique"),
    alp_deseq_genes=("alp_gene_id",lambda x:";".join(sorted(set(x)))),
    upregulated=("direction",lambda x:"upregulated" in set(x)),
    downregulated=("direction",lambda x:"downregulated" in set(x))).reset_index()
for direction in ["upregulated","downregulated"]:
    lists=mapped_de[mapped_de["direction"].eq(direction)].groupby("Orthogroup")["alp_gene_id"].agg(lambda x:";".join(sorted(set(x))))
    og_de[f"{direction}_genes"]=og_de["Orthogroup"].map(lists).fillna("")
both_direction_orthogroups=og_de[og_de["upregulated"]&og_de["downregulated"]].copy()
both_direction_orthogroups["treatment"]=BOTH_DIRECTIONS_POLICY
if BOTH_DIRECTIONS_POLICY=="exclude":
    mask=og_de["upregulated"]&og_de["downregulated"]; og_de.loc[mask,["upregulated","downregulated"]]=False
analysis_table=og_expression.merge(og_de,on="Orthogroup",how="inner").sort_values("Orthogroup").reset_index(drop=True)
analysis_table=analysis_table.merge(flags.rename(columns={c:f"high__{c}" for c in celltypes}),on="Orthogroup",validate="one_to_one")

retained_gene_count=int(gene_filter_diagnostics["background_eligible_gene"].sum())
excluded_broad_gene_count=int(gene_filter_diagnostics["excluded"].sum())
missing_stats_gene_count=int((~gene_filter_diagnostics["has_required_de_statistics"]).sum())
missing_stats_excluded_count=int((~gene_filter_diagnostics["has_required_de_statistics"]&~gene_filter_diagnostics["background_eligible_gene"]&~gene_filter_diagnostics["excluded"]).sum())
print(f"Expression orthogroups retained={len(og_expression):,}; excluded for no valid measurements={len(orthogroups_without_valid_expression):,}")
print(f"Broad filter excluded={excluded_broad_gene_count:,}; final background-eligible genes={retained_gene_count:,}")
print(f"Mapped genes missing required DE statistics={missing_stats_gene_count:,}; policy={MISSING_DE_STATS_BACKGROUND_POLICY}; excluded solely for missing statistics={missing_stats_excluded_count:,}")
print(f"Gene-cell-type assigned pairs exported={len(gene_celltype_table):,}; unassigned pairs included=False")
print(f"Eligible orthogroup background={len(analysis_table):,}; both directions={len(both_direction_orthogroups):,}; up={analysis_table.upregulated.sum():,}; down={analysis_table.downregulated.sum():,}")
display(threshold_table.head())
display(gene_celltype_table.head())

## Enrichment

One-sided Fisher tests use `alternative="greater"`. BH correction is applied jointly across all direction-by-cell-type tests.

In [ ]:
def bh(pvalues):
    p=np.asarray(pvalues,float); out=np.full(len(p),np.nan); valid=np.isfinite(p)
    if valid.any():
        idx=np.flatnonzero(valid); order=np.argsort(p[valid]); ranked=p[valid][order]
        q=np.minimum.accumulate((ranked*len(ranked)/np.arange(1,len(ranked)+1))[::-1])[::-1]
        out[idx[order]]=np.clip(q,0,1)
    return out

rows=[]
for direction in ["upregulated","downregulated"]:
    selected=analysis_table[direction].astype(bool)
    for celltype in celltypes:
        high=analysis_table[f"high__{celltype}"].fillna(False).astype(bool)
        a=int((selected&high).sum()); b=int((selected&~high).sum()); c=int((~selected&high).sum()); d=int((~selected&~high).sum())
        odds,p=fisher_exact([[a,b],[c,d]],alternative="greater")
        rows.append({"direction":direction,"celltype":celltype,"high_expression_quantile":HIGH_EXPRESSION_QUANTILE,"max_assigned_cell_types":MAX_ASSIGNED_CELL_TYPES,"padj_cutoff":PADJ_CUTOFF,"min_abs_log2_fold_change":MIN_ABS_LOG2_FOLD_CHANGE,"test_background_orthogroup_count":a+b+c+d,"selected_high":a,"selected_not_high":b,"not_selected_high":c,"not_selected_not_high":d,"selected_total":a+b,"high_total":a+c,"selected_fraction_high":a/(a+b) if a+b else np.nan,"background_fraction_high":(a+c)/(a+b+c+d) if a+b+c+d else np.nan,"odds_ratio":odds,"pvalue":p})
enrichment=pd.DataFrame(rows)
enrichment["fdr"]=bh(enrichment["pvalue"]); enrichment["correction_family"]=CORRECTION_FAMILY
enrichment["significantly_overrepresented"]=enrichment["fdr"].lt(FDR_CUTOFF)&enrichment["odds_ratio"].gt(1)
enrichment=enrichment.sort_values(["fdr","pvalue","direction","celltype"],na_position="last")
significant_enrichment=enrichment[enrichment["significantly_overrepresented"]].copy()
print(f"{len(enrichment)} tests; background={len(analysis_table):,} each; significant={len(significant_enrichment)}")
display(significant_enrichment)

## Diagnostics, visualization, and exports

Infinite odds ratios are retained in CSV output and capped only for plotting. Zero FDR values are handled safely. Assertions reconcile test and summary counts.

In [ ]:
mapping_summary=id_mapping_diagnostics.groupby(["source","status"],dropna=False).size().rename("count").reset_index()
pipeline_summary=pd.DataFrame([
{"section":"configuration","metric":"high_expression_quantile","value":HIGH_EXPRESSION_QUANTILE},
{"section":"configuration","metric":"max_assigned_cell_types","value":MAX_ASSIGNED_CELL_TYPES},
{"section":"configuration","metric":"padj_cutoff","value":PADJ_CUTOFF},
{"section":"configuration","metric":"min_abs_log2_fold_change","value":MIN_ABS_LOG2_FOLD_CHANGE},
{"section":"configuration","metric":"missing_de_stats_background_policy","value":MISSING_DE_STATS_BACKGROUND_POLICY},
{"section":"configuration","metric":"gene_celltype_unassigned_pairs_included","value":GENE_CELLTYPE_INCLUDE_UNASSIGNED},
{"section":"input","metric":"aep_expression_ids","value":len(expression)},
{"section":"input","metric":"deseq2_genes","value":len(deseq)},
{"section":"mapping","metric":"mapped_aep_expression_ids","value":len(mapped_expression_map)},
{"section":"mapping","metric":"mapped_deseq2_genes_unambiguous_before_broad_filter","value":len(mapped_de_ids)},
{"section":"broad_expression_filter","metric":"background_eligible_mapped_deseq2_genes","value":retained_gene_count},
{"section":"broad_expression_filter","metric":"excluded_broadly_expressed_genes","value":excluded_broad_gene_count},
{"section":"background","metric":"mapped_genes_missing_required_de_statistics","value":missing_stats_gene_count},
{"section":"background","metric":"genes_excluded_solely_for_missing_de_statistics","value":missing_stats_excluded_count},
{"section":"gene_celltype","metric":"assigned_gene_celltype_pairs","value":len(gene_celltype_table)},
{"section":"threshold","metric":"orthogroups_with_mapped_aep_expression","value":len(og_expression)},
{"section":"threshold","metric":"orthogroups_excluded_no_valid_expression","value":len(orthogroups_without_valid_expression)},
{"section":"enrichment","metric":"eligible_background_orthogroups","value":len(analysis_table)},
{"section":"enrichment","metric":"upregulated_orthogroups_after_policy","value":int(analysis_table.upregulated.sum())},
{"section":"enrichment","metric":"downregulated_orthogroups_after_policy","value":int(analysis_table.downregulated.sum())},
{"section":"ambiguity","metric":"both_direction_orthogroups_before_policy","value":len(both_direction_orthogroups)},
{"section":"enrichment","metric":"tests","value":len(enrichment)},
{"section":"enrichment","metric":"significant_tests","value":len(significant_enrichment)}])
assert analysis_table["Orthogroup"].is_unique
assert len(enrichment)==2*len(celltypes)
assert (enrichment["test_background_orthogroup_count"]==len(analysis_table)).all()
assert not set(orthogroups_without_valid_expression["Orthogroup"]) & set(analysis_table["Orthogroup"])
assert gene_celltype_table["high_expression_assigned"].all()
assert not gene_celltype_table["unassigned_pairs_included"].any()

if significant_enrichment.empty:
    print(f"No significant results at FDR < {FDR_CUTOFF}; no plot written.")
else:
    p=significant_enrichment.copy(); finite=p.loc[np.isfinite(p.odds_ratio),"odds_ratio"]; cap=max(2.0,float(finite.max())*1.1) if len(finite) else 10.0
    p["plot_odds"]=p.odds_ratio.replace(np.inf,cap).fillna(0); p["label"]=p.direction+": "+p.celltype
    p=p.sort_values(["direction","fdr"],ascending=[True,False])
    fig,ax=plt.subplots(figsize=(10,max(4,.35*len(p)))); ax.barh(p.label,p.plot_odds,color=p.direction.map({"upregulated":"#C44E52","downregulated":"#4C72B0"}))
    ax.set_xlabel("Fisher odds ratio (infinite capped for display)"); ax.set_title(f"High AEP expression enrichment (q={HIGH_EXPRESSION_QUANTILE:.2f})")
    fig.tight_layout(); fig.savefig(OUTPUT_DIR/"significant_celltype_high_expression_enrichment.png",dpi=300,bbox_inches="tight"); plt.show()

parts=[]
for row in significant_enrichment.itertuples(index=False):
    mask=analysis_table[row.direction]&analysis_table[f"high__{row.celltype}"]
    x=analysis_table.loc[mask,["Orthogroup","alp_deseq_gene_count","alp_deseq_genes",f"{row.direction}_genes",row.celltype]].copy()
    x=x.rename(columns={f"{row.direction}_genes":"directional_alp_genes",row.celltype:"mean_aep_expression"}); x.insert(0,"celltype",row.celltype); x.insert(0,"direction",row.direction); parts.append(x)
contributors=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(columns=["direction","celltype","Orthogroup","alp_deseq_gene_count","alp_deseq_genes","directional_alp_genes","mean_aep_expression"])
exports={"orthogroup_analysis_table.csv":analysis_table,"high_expression_thresholds.csv":threshold_table,"id_mapping_diagnostics.csv":id_mapping_diagnostics,"mapping_summary.csv":mapping_summary,"ambiguous_alp_gene_orthogroup_mappings.csv":ambiguous_gene_mappings,"invalid_alp_protein_isoform_ids.csv":invalid_alp_protein_ids,"invalid_aep_orthofinder_ids.csv":invalid_aep_orthofinder_ids,"orthogroups_significant_in_both_directions.csv":both_direction_orthogroups,"alp_gene_broad_expression_filter_diagnostics.csv":gene_filter_diagnostics,"alp_gene_celltype_assignments.csv":gene_celltype_table,"orthogroups_without_valid_expression.csv":orthogroups_without_valid_expression,"invalid_aep_expression_values.csv":invalid_expression_values,"invalid_deseq2_numeric_values.csv":invalid_deseq_values,"pipeline_summary.csv":pipeline_summary,"celltype_high_expression_enrichment_all_tests.csv":enrichment,"celltype_high_expression_enrichment_significant.csv":significant_enrichment,"orthogroups_contributing_to_significant_enrichment.csv":contributors}
for name,df in exports.items(): df.to_csv(OUTPUT_DIR/name,index=False)
for name,df in exports.items():
    if not (OUTPUT_DIR/name).is_file(): raise RuntimeError(f"Missing export {name}")
print(f"Wrote {len(exports)} CSV files to {OUTPUT_DIR}")
display(pipeline_summary)

In [ ]:
# Export plot-ready values, affected genes for plotted results, and all mapped ALP gene-cell-type assignments.
if significant_enrichment.empty:
    data_to_plot = significant_enrichment.copy()
    data_to_plot['plot_odds'] = pd.Series(dtype=float)
    data_to_plot['label'] = pd.Series(dtype=str)
else:
    data_to_plot = significant_enrichment.copy()
    finite_plot_odds = data_to_plot.loc[np.isfinite(data_to_plot['odds_ratio']), 'odds_ratio']
    plot_cap = max(2.0, float(finite_plot_odds.max()) * 1.1) if len(finite_plot_odds) else 10.0
    data_to_plot['plot_odds'] = data_to_plot['odds_ratio'].replace(np.inf, plot_cap).fillna(0)
    data_to_plot['label'] = data_to_plot['direction'] + ': ' + data_to_plot['celltype']
    data_to_plot = data_to_plot.sort_values(['direction', 'fdr'], ascending=[True, False]).reset_index(drop=True)

affected_gene_rows = []
for row in contributors.itertuples(index=False):
    genes = [gene for gene in str(row.directional_alp_genes).split(';') if gene]
    affected_gene_rows.extend(
        {'direction': row.direction, 'celltype': row.celltype, 'alp_gene_id': gene}
        for gene in genes
    )
affected_genes_long = pd.DataFrame(
    affected_gene_rows, columns=['direction', 'celltype', 'alp_gene_id']
).drop_duplicates()

if affected_genes_long.empty:
    affected_genes_by_celltype = pd.DataFrame(
        columns=['direction', 'celltype', 'affected_gene_count', 'affected_alp_genes']
    )
else:
    affected_genes_by_celltype = (
        affected_genes_long.groupby(['direction', 'celltype'], as_index=False)
        .agg(
            affected_gene_count=('alp_gene_id', 'nunique'),
            affected_alp_genes=('alp_gene_id', lambda x: ';'.join(sorted(set(x)))),
        )
        .sort_values(['direction', 'celltype'])
        .reset_index(drop=True)
    )

mapped_alp_genes_by_celltype = (
    gene_celltype_table[[
        'celltype', 'alp_gene_id', 'Orthogroup', 'log2FoldChange', 'padj', 'direction',
        'mean_aep_expression', 'threshold', 'assigned_cell_type_count', 'excluded',
        'exclusion_reason', 'background_eligible_gene', 'background_exclusion_reason',
    ]]
    .drop_duplicates(['celltype', 'alp_gene_id', 'Orthogroup'])
    .sort_values(['celltype', 'alp_gene_id', 'Orthogroup'])
    .reset_index(drop=True)
)

assert len(data_to_plot) == len(significant_enrichment)
assert not affected_genes_long.duplicated(['direction', 'celltype', 'alp_gene_id']).any()
assert not mapped_alp_genes_by_celltype.duplicated(['celltype', 'alp_gene_id', 'Orthogroup']).any()

additional_exports = {
    'data_to_plot.csv': data_to_plot,
    'affected_genes_by_celltype.csv': affected_genes_by_celltype,
    'mapped_alp_genes_by_celltype.csv': mapped_alp_genes_by_celltype,
}
for name, df in additional_exports.items():
    df.to_csv(OUTPUT_DIR / name, index=False)
    if not (OUTPUT_DIR / name).is_file():
        raise RuntimeError(f'Missing export {name}')

print(f'Wrote {len(additional_exports)} additional CSV files to {OUTPUT_DIR}')
print(f'Plot rows={len(data_to_plot):,}; affected gene groups={len(affected_genes_by_celltype):,}; mapped gene-cell-type rows={len(mapped_alp_genes_by_celltype):,}')
display(data_to_plot)
display(affected_genes_by_celltype)
display(mapped_alp_genes_by_celltype.head())

## Interpretation

Each enrichment result tests whether **orthogroups** containing significant up- or downregulated ALP genes are overrepresented among AEP orthogroups at or above the configured cell-type expression quantile. Thresholds are calculated from all mapped-expression orthogroups that have at least one valid cell-type measurement. Orthogroups with no valid expression measurements are excluded and exported separately. Because threshold membership is inclusive, ties can make the realized high-expression fraction exceed the nominal top fraction; both are reported per cell type.

Before aggregation, mapped genes above the broad-expression assignment maximum are excluded. Under the default missing-statistics policy, genes lacking either `log2FoldChange` or `padj` are also excluded from the enrichment background; the policy and affected counts are recorded. An eligible orthogroup must retain at least one background-eligible gene.

`alp_gene_celltype_assignments.csv` contains one row for every **assigned** gene–cell-type pair, combining `log2FoldChange`, `padj`, direction, orthogroup expression, threshold, and filter/background status. Unassigned gene–cell-type pairs are intentionally omitted; genes with zero assignments remain visible in the gene-level broad-filter diagnostic.

A significant enrichment is an orthogroup-level association, not evidence that every paralog is regulated or functionally equivalent.

In [ ]:
female_germlinecell_ogs = gene_celltype_table[(gene_celltype_table["celltype"] == "I_FemGC") & (gene_celltype_table["excluded"] == False) & (gene_celltype_table["log2FoldChange"] <= 0.0)]

In [ ]:
female_germlinecell_ogs[female_germlinecell_ogs["Orthogroup"] == "OG0001257"]

In [ ]:
len(female_germlinecell_ogs)

In [ ]:
orthogroups[orthogroups["Orthogroup"].isin(female_germlinecell_ogs["Orthogroup"].to_list())]

In [ ]:
mapped_alp_genes_by_celltype[(mapped_alp_genes_by_celltype["celltype"] == "I_FemGC") & (mapped_alp_genes_by_celltype["excluded"] == False)]